In [1]:
import torch
import torch.nn as nn
import math
import tiktoken

In [2]:
text1 = "I live in France ok"
text2 = "I am from India ."
text3 = "France is a good country" 


tokenizer = tiktoken.get_encoding("gpt2")

text1_enc = torch.tensor(tokenizer.encode(text1))
text2_enc = torch.tensor(tokenizer.encode(text2))
text3_enc = torch.tensor(tokenizer.encode(text3))


text1_enc, text2_enc, text3_enc

input = torch.stack([text1_enc, text2_enc, text3_enc])
input.shape



torch.Size([3, 5])

In [3]:
GPT_CONFIG_124M = {
                    "vocab_size": 50257,        # Vocabulary size
                    "context_length": 1024,     # Context length
                    "emb_dim": 768,             # Embedding dimension
                    "n_heads": 12,              # Number of attention heads
                    "n_layers": 12,             # Number of layers
                    "drop_rate": 0.1,           # Dropout rate
                    "qkv_bias": False           # Query-Key-Value bias
                    }

In [4]:
config = {
                    "vocab_size": 50257,        # Vocabulary size
                    "context_length": 1024,     # Context length
                    "emb_dim": 768,             # Embedding dimension
                    "n_heads": 12,              # Number of attention heads
                    "n_layers": 12,             # Number of layers
                    "drop_rate": 0.1,           # Dropout rate
                    "qkv_bias": False           # Query-Key-Value bias
                    }

In [5]:
embedding_layer = nn.Embedding(config["vocab_size"], config["emb_dim"])
emb_layer_out = embedding_layer(input)
emb_layer_out.shape


torch.Size([3, 5, 768])

In [6]:
seq_len = input.shape[-1]
pos_embedding = nn.Embedding(config["context_length"], config["emb_dim"])
pos_emb_out = pos_embedding(torch.arange(seq_len))
pos_emb_out.shape

torch.Size([5, 768])

In [7]:
input_tr = emb_layer_out+pos_emb_out
input_tr.shape


torch.Size([3, 5, 768])

In [8]:
class LayerNorm(nn.Module):

    def __init__(self, emb_dim):
        super().__init__()

        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))



    def forward(self, x):

        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True)

        x_norm = (x-mean)/torch.sqrt(var+self.eps)

        return self.scale * x_norm + self.shift


In [9]:
ln = LayerNorm(config["emb_dim"])
out1 = ln(input_tr)
out1.shape
# out1.shape, out1.mean(dim=-1), out1.var(dim=-1)

torch.Size([3, 5, 768])

In [10]:
class GELU(nn.Module):

    def __init__(self):
        super().__init__()

        self.coeff = math.sqrt(2/math.pi)

    def forward(self,x):

        return 0.5 * x * (1 + torch.tanh( self.coeff * ( x + 0.044715 * x.pow(3) ) ))

In [11]:
class FeedForward(nn.Module):

    def __init__(self, emb_dim):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(emb_dim, 4*emb_dim),
            GELU(),
            nn.Linear(4*emb_dim, emb_dim)
        )

    def forward(self,x):
        return self.layers(x)

In [12]:
ff = FeedForward(config["emb_dim"])
out2 = ff(out1)
out1.shape, out2.shape

(torch.Size([3, 5, 768]), torch.Size([3, 5, 768]))

In [13]:
class CausalMultiHeadAttention(nn.Module):

    def __init__(self,d_in, d_out, emb_dim, n_head, context_len, drop_rate=0.0, qkv_bias=False):
        super().__init__()

        assert emb_dim % n_head ==0 , "Embedding dimention must be divisable by number of attention heads"

        self.head_dim = emb_dim//n_head
        self.n_heads = n_head
        self.emb_dim = emb_dim

        self.q_proj = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.k_proj = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.v_proj = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.head_proj = nn.Linear(d_out, d_in, bias=qkv_bias)

        self.dropout = nn.Dropout(drop_rate)

        self.register_buffer(
            "mask",

            torch.tril(torch.ones(context_len, context_len)).bool()

            )

        self.scale = math.sqrt(self.head_dim)

    def forward(self,x):


        batch_size, seq_len, emb_dim = x.shape

        Q = self.q_proj(x).reshape(batch_size, seq_len, self.n_heads, self.head_dim).transpose(-3,-2)
        K = self.k_proj(x).reshape(batch_size, seq_len, self.n_heads, self.head_dim).transpose(-3,-2)
        V = self.v_proj(x).reshape(batch_size, seq_len, self.n_heads, self.head_dim).transpose(-3,-2)
        # shape here batch_size, self.n_heads, seq_len, self.head_dim
        attention_scores = Q @ K.transpose(-2,-1)

        mask = self.mask[:seq_len, :seq_len]

        attention_scores = attention_scores.masked_fill(~mask, float("-inf"))

        # attention_scores = self.dropout(attention_scores)

        attention_weight = torch.softmax(attention_scores/self.scale, dim=-1)

        attention_weight = self.dropout(attention_weight)

        att_out = attention_weight @ V

        att_out = att_out.transpose(-3,-2).reshape(batch_size,seq_len,-1)

        out = self.head_proj(att_out)

        return out



In [14]:
class TransformerBlock(nn.Module):

    def __init__(self, config):
        super().__init__()

        emb_dim = config["emb_dim"]

        self.layernorm1 = LayerNorm(emb_dim)
        self.mha = CausalMultiHeadAttention(emb_dim, emb_dim, emb_dim, config["n_heads"], config["context_length"])
        self.layernorm2 = LayerNorm(emb_dim)
        self.ff = FeedForward(emb_dim)


    def forward(self,x):

        shortcut = x
        x = self.layernorm1(x)
        x = self.mha(x)


        x = x + shortcut

        shortcut = x

        x = self.layernorm2(x)
        x = self.ff(x)

        x = x + shortcut

        return x

In [15]:
tfb = TransformerBlock(config)
# print(TransformerBlock)

out3 = tfb(input_tr)

print(input_tr.shape, out3.shape)

torch.Size([3, 5, 768]) torch.Size([3, 5, 768])


In [16]:
class Lumiere(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.token_emb = nn.Embedding(config["vocab_size"], config["emb_dim"])
        self.pos_emb = nn.Embedding(config["context_length"], config["emb_dim"])

        self.tr_blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config["n_layers"])])

        self.final_layer_norm = LayerNorm(config["emb_dim"])

        self.out_head = nn.Linear(config["emb_dim"], config["vocab_size"])

        self.dropout = nn.Dropout(config["drop_rate"])

    def forward(self,x):

        batch_size, seq_len = x.shape

        x_tk_emb = self.token_emb(x)
        x_pos_emb = self.pos_emb(torch.arange(seq_len))

        x = x_tk_emb + x_pos_emb

        x = self.dropout(x)

        for block in self.tr_blocks:

            x = block(x)

        x = self.final_layer_norm(x)
        x = self.out_head(x)

        return x


In [17]:
print(input)
print(input.shape)


model = Lumiere(config)

out_final = model(input)

print(input.shape, out_final.shape)

tensor([[   40,  2107,   287,  4881, 12876],
        [   40,   716,   422,  3794,   764],
        [28572,   318,   257,   922,  1499]])
torch.Size([3, 5])
torch.Size([3, 5]) torch.Size([3, 5, 50257])


In [18]:
def generate_text(model, start_tokens_batch, max_generate_len=3):

    model.eval()

    for _ in range(max_generate_len):

        # print(start_tokens_batch.shape)

        model_output_logits = model(start_tokens_batch[:,-config["context_length"]:])
        # print(model_output_logits.shape)

        last_output_logits = model_output_logits[:,-1,:]
        # print(last_output_logits.shape)

        pred_id = torch.argmax(last_output_logits, dim=-1, keepdim=True)
        # print(pred_id)
        # print(pred_id.shape)

        start_tokens_batch = torch.cat([start_tokens_batch, pred_id], dim=-1)

        # print(start_tokens_batch)
        # print(start_tokens_batch.shape)

    return start_tokens_batch



model = Lumiere(config)
out = model(input)

out_tkn= generate_text(model, input)

print(input)
print(out_tkn)

[print(tokenizer.decode(p.tolist())) for p in input]
[print(tokenizer.decode(p.tolist())) for p in out_tkn]


tensor([[   40,  2107,   287,  4881, 12876],
        [   40,   716,   422,  3794,   764],
        [28572,   318,   257,   922,  1499]])
tensor([[   40,  2107,   287,  4881, 12876, 45151,  6292, 14625],
        [   40,   716,   422,  3794,   764, 11005, 39667, 26894],
        [28572,   318,   257,   922,  1499, 46154, 49684, 49008]])
I live in France ok
I am from India .
France is a good country
I live in France ok725 acceptedresa
I am from India . eliminate418 [...]
France is a good country Karachi MISS mortals


[None, None, None]